In [ ]:
import sys, os
sys.path.insert(0, r"C:\Users\Utente\Desktop\progetto vscode\src")

In [1]:
import numpy as np
import networkx as nx
import ot
import matplotlib.pyplot as plt
import seaborn as sns
import random
from EDRep import NodeEmbedding  
from functions import *


Create a graph with sbm and with it do some sanity checks.

In [2]:
class_sizes = [100, 100, 100]

matrix_1 = np.where(np.eye(3, dtype=bool), 0.8, 0.1)
matrix_2 = np.where(np.eye(3, dtype=bool), 0.6, 0.3)

G_1 = nx.stochastic_block_model(class_sizes, matrix_1 , seed=42)
G_2 = nx.stochastic_block_model(class_sizes, matrix_1 , seed=43)
G_3 = nx.stochastic_block_model(class_sizes, matrix_2 , seed=44)

In [3]:
embedding_dim= 16
def partition(class_sizes):
    part = []
    start = 0
    for i in class_sizes:
        part.append(list(range(start, start + i)))
        start += i
    return part

partitions= partition(class_sizes)
emb_1 = node_embedding(G_1, partitions, embedding_dim)
emb_2 = node_embedding(G_2, partitions, embedding_dim)
emb_3 = node_embedding(G_3, partitions, embedding_dim)


In [7]:
dist_G1_G1, _, _ = w2_distance_classes(emb_1, emb_1)
dist_G1_G2, _, _ = w2_distance_classes(emb_1, emb_2)
dist_G2_G1, _, _ = w2_distance_classes(emb_2, emb_1)
dist_G1_G3, _, _ = w2_distance_classes(emb_1, emb_3)
dist_G2_G3, _, _ = w2_distance_classes(emb_3, emb_2)

In [9]:
dist_G1_G2+dist_G2_G3>dist_G1_G3

np.True_

In [ ]:
dist_G1_G1, _, _ = w2_distance_classes(emb_1, emb_1)
dist_G1_G2, _, _ = w2_distance_classes(emb_1, emb_2)
dist_G2_G1, _, _ = w2_distance_classes(emb_2, emb_1)
dist_G1_G3, _, _ = w2_distance_classes(emb_1, emb_3)

print(f"1. Identity d(G_1, G_1): {dist_G1_G1:.4f}")
print(f"s. Symmetry d(G_1, G_2): {dist_G1_G2:.4f} and d(G_2, G_1): {dist_G2_G1:.4f} ")
print(f"3. Distance between similar graphs d(G_1, G_2): {dist_G1_G2:.4f}")
print(f"4. Distance between very different graphs d(G_1, G_3): {dist_G1_G3:.4f}")
print(f"5. Triangle Inequality: {dist_G1_G3:.4f}")


 We show that mixing the nodes of the same class doesn't change the output while changing them betwwn different class does

In [ ]:
def intra_class_perm(partitions):
    perm_partitions = []
    for p in partitions:
        perm_partitions.append(list(np.random.permutation(p)))
    return perm_partitions


perm_partition= intra_class_perm(partitions)
emb_1_perm = node_embedding(G_1, perm_partition, embedding_dim)
dist_G1_G1perm, _, _ = w2_distance_classes(emb_1, emb_1_perm)
print(dist_G1_G1perm)


In [10]:
def inter_class_perm(G,partitions):
    class_size = [len(p) for p in partitions]
    shuffled_nodes = list(np.random.permutation(list(G.nodes())))

    new_partitions = []
    for i in partition(class_size):
        new_class = [shuffled_nodes[i] for i in i]
        new_partitions.append(new_class)
        
    return new_partitions

inter_perm_partition= inter_class_perm(G_1,partitions)
emb_1_inter_perm = node_embedding(G_1, inter_perm_partition, embedding_dim)
dist_G1_G1inter_perm, _, _ = w2_distance_classes(emb_1, emb_1_inter_perm)
print(dist_G1_G1inter_perm)

1.8817852564506143


In [11]:

inter_perm_partition3= inter_class_perm(G_3,partitions)
emb_3_inter_perm = node_embedding(G_3, inter_perm_partition3, embedding_dim)
dist_G3_G3inter_perm, _, _ = w2_distance_classes(emb_3, emb_3_inter_perm)
print(dist_G3_G3inter_perm)

0.37501121092854206


Matched case


In [ ]:
def rand_graph_one_in_every_part(n, prob):
   
    G_random = nx.erdos_renyi_graph(n, prob)
    partition_random = [[nodes] for nodes in G_random.nodes()]   
    return G_random, partition_random


In [ ]:
G1, part1 = rand_graph_one_in_every_part(500,0.3)
G2, part2 = rand_graph_one_in_every_part(500,0.2)

In [ ]:
Embedding1=node_embedding(G1, part1,32)
Embedding2=node_embedding(G2, part2,32)

dist_G1G2, _, _= w2_distance_classes(Embedding1,Embedding2)
print(dist_G1G2)

In [ ]:
x=np.array([e[0] for e in Embedding1])
y=np.array([e[0] for e in Embedding2])
np.linalg.norm(x@x.T-y@y.T)/math.sqrt(2)